# Sesión 5 — Clasificación: regresión logística, k-NN… y el accuracy que miente

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 2 — Modelos supervisados

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar por qué una *y* de sí/no necesita una **sigmoide** y entrenar una **regresión logística** que entrega probabilidades.
2. Leer una **matriz de confusión** y explicar por qué el **accuracy engaña** cuando una clase es rara; calcular **recall** y **precisión**.
3. Ajustar el **umbral de decisión** y el **balance de clases** según lo que cuesta cada error en el negocio.
4. Entrenar **k-NN**, explicar por qué necesita variables escaladas y qué controla `k`.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: dos modelos de clasificación, y por qué un número de evaluación puede mentir.
```

## Cómo usar este notebook
🔍 **Predice** antes de ejecutar · ✏️ **Completa** (andamiaje medio) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Datos de hoy: Bank Marketing** (Moro, Cortez y Rita, 2014; UCI Machine Learning Repository). 45 211 llamadas de telemercadeo de un banco portugués ofreciendo un depósito a término. La pregunta: **¿este cliente va a decir que sí?** Es el mismo problema que tiene cualquier banco o empresa de telecomunicaciones en Colombia que hace campañas por teléfono.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["modelo", "evaluación"], modulo=2, subtitulo="HOY: clasificar… y descubrir que el accuracy puede mentir");

## 1. Los datos: una campaña de telemercadeo

| Grupo | Columnas |
|---|---|
| Cliente | `age`, `job`, `marital`, `education`, `default` (mora), `balance` (saldo promedio, €), `housing` (crédito de vivienda), `loan` (crédito personal) |
| Esta campaña | `contact` (tipo de teléfono), `day`, `month` (último contacto), `campaign` (llamadas en esta campaña) |
| Campañas anteriores | `pdays` (días desde el último contacto; **−1 = nunca contactado**), `previous`, `poutcome` (resultado anterior) |
| **y** | `deposit`: ¿suscribió el depósito? (`yes` / `no`) |

### 🔍 Predice
¿Qué porcentaje de los clientes llamados cree que dijo **sí**?

**Tu predicción:** ____ %

In [ ]:
bank = pd.read_csv("bank.csv")
print(bank.shape)
print(bank["deposit"].value_counts(normalize=True).round(3))

**11,7 %.** Nueve de cada diez llamadas terminan en "no". Esta proporción va a ser la protagonista de la sesión.

### ✏️ Completa — la Sesión 2 en dos líneas
`pdays = -1` y `poutcome = "unknown"` son **vacíos disfrazados** (ficha de la Sesión 2: valor imposible y texto que significa "no sé"). ¿Qué fracción de las filas los tiene?

In [ ]:
print("pdays = -1:", ...)          # TODO: fracción de filas con pdays == -1
print("poutcome = unknown:", ...)  # TODO: fracción de filas con poutcome == "unknown"

**82 %** en los dos, y no es casualidad: son los clientes que **nunca habían sido contactados**, así que no tienen campaña anterior. Aquí el vacío **informa** (lo vimos en la Sesión 2) y lo dejamos como su propia categoría (`poutcome_unknown`).

Una decisión de entrada: dejamos por fuera la columna `duration` (duración de la llamada, en segundos). La retomamos en la Sesión 6.

## 2. ¿Por qué no una recta? La regresión logística

En la Sesión 4 ajustamos rectas a números. Si la *y* es 0/1 y le ajustamos una recta, la "probabilidad" se sale de [0, 1] en los extremos. La solución es **doblar la recta** con una curva en forma de S, la **sigmoide**.

In [ ]:
fg.sigmoide_vs_recta();

### 📐 Fundamento

La logística calcula primero lo mismo que una regresión lineal, una suma $z$, y luego la pasa por la sigmoide:

$$z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p \qquad\qquad p(\text{sí}) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

- $z$ puede valer cualquier número; $\sigma(z)$ siempre queda entre 0 y 1.
- Si $z = 0$, $p = 0{,}5$: ahí está la **frontera** donde el modelo cambia de "no" a "sí".
- Cada $\beta_j$ dice cuánto empuja la variable $j$ hacia "sí" (positivo) o hacia "no" (negativo).

¿Qué minimiza? No la suma de cuadrados sino el **log-loss** (entropía cruzada):

$$L = -\frac{1}{n}\sum_i \big[\,y_i \log p_i + (1 - y_i)\log(1 - p_i)\,\big]$$

Léala así: si el cliente dijo sí ($y=1$) y el modelo le dio $p=0{,}01$, el castigo es $-\log(0{,}01) \approx 4{,}6$; si le dio $p=0{,}9$, apenas $0{,}1$. **Castiga sobre todo estar seguro y equivocado.**

<details><summary><b>Para quien quiera más:</b> odds y log-odds</summary>

$z$ es el logaritmo de los *odds*: $z = \log\frac{p}{1-p}$. Por eso $e^{\beta_j}$ se lee como un *odds ratio*: cuántas veces se multiplican los odds de "sí" por cada unidad que sube $x_j$. Con variables escaladas, "una unidad" es una desviación estándar.
</details>

Con dos variables, la frontera $z = 0$ es una **recta** en el plano, y el color de fondo es la probabilidad:

In [ ]:
fg.frontera_logistica_2d();

![Dentro del .fit() — regresión logística y k-NN](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S05/img/dentro_del_fit_logistica_knn.png)

Fíjese en la diferencia con la Sesión 4: la logística **no tiene fórmula cerrada**. Su `.fit()` siempre es iterativo, como el descenso de gradiente que usted programó.

## 3. Entrenar la logística

Preparación (todo ya conocido): *one-hot* con `drop_first`, separar con `stratify` (Sesión 3) y **escalar con el escalador ajustado solo en entrenamiento**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

escalador = StandardScaler().fit(X_train)
Z_train, Z_test = escalador.transform(X_train), escalador.transform(X_test)
print("X:", X.shape, "| entrenamiento:", Z_train.shape, "| prueba:", Z_test.shape)

Primero, un detalle del optimizador. Entrenemos **sin escalar** y con pocas iteraciones:

In [ ]:
from sklearn.linear_model import LogisticRegression

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    LogisticRegression(max_iter=100).fit(X_train, y_train)
print("Avisos:", [type(a.message).__name__ for a in avisos] or "ninguno")

Ese `ConvergenceWarning` significa *"me quedé sin pasos antes de llegar al fondo"*. Es el tazón alargado de la Sesión 4: con variables en escalas muy distintas (`balance` en miles, `campaign` en unidades), el optimizador zigzaguea. Con las variables escaladas, llega.

### ✏️ Completa
Entrene una `LogisticRegression(max_iter=2000)` con las variables **escaladas** de entrenamiento.

In [ ]:
logistica = ...   # TODO: LogisticRegression(max_iter=2000) entrenada con Z_train, y_train
print("iteraciones usadas:", logistica.n_iter_[0])

### 🔍 Predice
¿Qué **accuracy** cree que tendrá la logística en prueba?

**Tu predicción:** ____

In [ ]:
from sklearn.metrics import accuracy_score

pred = logistica.predict(Z_test)
print(f"Accuracy de la regresión logística: {accuracy_score(y_test, pred):.3f}")

**0,894.** El modelo acierta en el 89 % de los clientes. ¿Lo compraría el banco?

## 4. 💥 El 89 % que no sirve para nada (casi)

### 🔍 Predice
El banco quiere el modelo para **encontrar a los clientes que van a decir sí**. En prueba hay 1 058 de esos clientes. ¿A cuántos cree que identifica el modelo (predice "sí" y efectivamente dijeron sí)?

**Tu predicción:** ____ de 1 058

In [ ]:
from sklearn.dummy import DummyClassifier

base = DummyClassifier(strategy="most_frequent").fit(Z_train, y_train)
fg.matrices_confusion({"línea base (siempre 'no')": (y_test, base.predict(Z_test)),
                       "regresión logística": (y_test, pred)});

![Accuracy, recall y precisión](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S05/img/accuracy_recall_precision.png)

**Recall** responde la pregunta del banco: *de los que sí iban a comprar, ¿a cuántos encontré?* **Precisión** responde la del call center: *de los que llamé, ¿cuántos dijeron sí?*

## 5. Dos formas de arreglarlo

### 5.1 Mover el umbral
`.predict()` dice "sí" cuando la probabilidad es ≥ 0,5. Pero ese 0,5 es una **convención**, no una ley: el banco puede llamar a todos los que tengan probabilidad ≥ 0,2.

### 🔍 Predice
Si bajamos el umbral de 0,5 a 0,2, ¿qué pasa con el recall? ¿Y con el accuracy?

**Tu predicción:** recall ____ · accuracy ____

In [ ]:
proba = logistica.predict_proba(Z_test)[:, 1]      # probabilidad de "sí" de cada cliente
fg.umbral_efecto(y_test, proba, marcar=(0.5, 0.2));

### ✏️ Completa — prediga con umbral 0,2

In [ ]:
from sklearn.metrics import recall_score, precision_score

pred_02 = ...   # TODO: 1 si proba >= 0.2, 0 si no (pista: (proba >= 0.2).astype(int))
for nombre, p in [("umbral 0,5", pred), ("umbral 0,2", pred_02)]:
    print(f"{nombre}: accuracy {accuracy_score(y_test, p):.3f} · recall {recall_score(y_test, p):.3f} · precisión {precision_score(y_test, p):.3f}")

### 5.2 Balancear las clases en el entrenamiento
La otra forma es decirle al `.fit()` que un error en un "sí" pesa más que un error en un "no": `class_weight="balanced"` le da a cada clase un peso inversamente proporcional a su frecuencia (cada "sí" pesa unas 7,5 veces más).

In [ ]:
logistica_bal = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Z_train, y_train)
pred_bal = logistica_bal.predict(Z_test)
fg.matrices_confusion({"logística, umbral 0,5": (y_test, pred),
                       "logística, umbral 0,2": (y_test, pred_02),
                       "logística balanceada": (y_test, pred_bal)});

### ¿Cuál es mejor? Depende de lo que cuesta cada error
Ninguna métrica decide sola. Supongamos que cada llamada le cuesta al banco **20 000 pesos** (tiempo del asesor) y cada depósito suscrito le deja **100 000**. La ganancia de cada estrategia sobre los 9 043 clientes de prueba:

In [ ]:
COSTO_LLAMADA, GANANCIA_SI = 20_000, 100_000

def ganancia(p):
    llamadas = p.sum()
    aciertos = ((p == 1) & (y_test.values == 1)).sum()
    return aciertos * GANANCIA_SI - llamadas * COSTO_LLAMADA

estrategias = {"no llamar a nadie (línea base)": base.predict(Z_test), "llamar a todos": np.ones(len(y_test), int),
               "logística, umbral 0,5": pred, "logística, umbral 0,2": pred_02, "logística balanceada": pred_bal}
pd.DataFrame({n: {"llamadas": int(p.sum()), "accuracy": round(accuracy_score(y_test, p), 3),
                  "recall": round(recall_score(y_test, p), 3), "ganancia (millones COP)": round(ganancia(p) / 1e6, 1)}
              for n, p in estrategias.items()}).T

## 6. Una fila entra, una predicción sale

La logística se explica como la regresión de la Sesión 4, en dos pasos: **sumar** contribuciones (en la escala de los log-odds) y **pasar** la suma por la sigmoide. Este es el cliente de prueba al que el modelo le dio más probabilidad de decir sí, entre los que de verdad dijeron sí:

In [ ]:
positivos = np.where(y_test.values == 1)[0]
i = int(positivos[np.argmax(proba[positivos])])
fg.logistica_una_fila(logistica, Z_test[i], X.columns, real=int(y_test.iloc[i]));

Lo que más empuja hacia "sí" en general:

In [ ]:
coefs = pd.Series(logistica.coef_[0], index=X.columns)
pd.concat([coefs.sort_values().head(5), coefs.sort_values().tail(5)]).round(2)

## 7. Otro modelo, otra idea: k vecinos más cercanos (k-NN)

k-NN no aprende ninguna fórmula: **guarda los datos** y, para un cliente nuevo, busca los $k$ clientes más parecidos y pregunta qué hicieron ellos.

In [ ]:
fg.knn_vecinos(k=7);

### 📐 Fundamento
"Parecido" se mide con la **distancia euclidiana** entre dos clientes $a$ y $b$:

$$d(a, b) = \sqrt{\sum_j (a_j - b_j)^2}$$

La probabilidad de "sí" es simplemente la fracción de vecinos que dijo sí. Mire la ficha "Dentro del `.fit()`": k-NN **no optimiza nada**; todo el trabajo pasa en `.predict()`, que tiene que medir la distancia a los 36 000 clientes de entrenamiento.

### 🔍 Predice
En la suma de la distancia, `balance` (saldo, en euros) puede valer 10 000 y `age` (edad) 40. Si **no** escalamos, ¿qué variable decide quién es "vecino" de quién?

**Tu predicción:** ____

In [ ]:
fg.knn_escala(bank, "age", "balance", k=15);

Sin escalar, los "vecinos" de un cliente de 37 años tienen de 27 a 60 años: solo se parecen en el saldo. Es la promesa de la Sesión 2 cumplida: **los modelos que miden distancias necesitan variables escaladas.**

### ✏️ Completa — k-NN con y sin escalar
Entrene dos `KNeighborsClassifier(n_neighbors=25)`: uno con `Z_train` (escalado) y otro con `X_train` (sin escalar). Compare el recall.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_z = ...   # TODO: KNeighborsClassifier(n_neighbors=25) entrenado con Z_train
knn_x = ...   # TODO: el mismo, entrenado con X_train.values (sin escalar)
for nombre, m, Xp in [("escalado", knn_z, Z_test), ("sin escalar", knn_x, X_test.values)]:
    p = m.predict(Xp)
    print(f"k-NN {nombre:<12} accuracy {accuracy_score(y_test, p):.3f} · recall {recall_score(y_test, p):.3f}")

### La perilla de k-NN: `k`

### 🔍 Predice
Con `k = 1`, ¿qué accuracy tendrá k-NN **en entrenamiento**? ¿Por qué?

**Tu predicción:** ____

In [ ]:
fg.knn_k_efecto(ks=(1, 5, 25, 101));

In [ ]:
filas = []
for k in (1, 5, 25, 101):
    m = KNeighborsClassifier(n_neighbors=k).fit(Z_train, y_train)
    p = m.predict(Z_test)
    filas.append({"k": k, "accuracy": round(accuracy_score(y_test, p), 3), "recall": round(recall_score(y_test, p), 3),
                  "precisión": round(precision_score(y_test, p), 3)})
pd.DataFrame(filas).set_index("k")

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 5) |
|---|---|
| Datos | clases desbalanceadas (11,7 % "sí") · `pdays = -1` y `poutcome = unknown`: vacíos que informan |
| Preprocesamiento | escalar es obligatorio para k-NN y ayuda a la convergencia de la logística |
| Modelo | **regresión logística** (sigmoide, log-loss, sin fórmula cerrada) · **k-NN** (no optimiza; `k`) |
| Evaluación | **matriz de confusión** · el accuracy engaña con clases desbalanceadas · **recall** y **precisión** · umbral y `class_weight` · la métrica la decide el negocio |
| Interpretación | contribuciones en log-odds → sigmoide · `poutcome_success` |

**Dentro del `.fit()` — la tabla crece:**

| | Árbol | Lineal | Logística | k-NN |
|---|---|---|---|---|
| Qué minimiza | Gini | residuales² | log-loss | nada |
| Cómo lo busca | exhaustiva y voraz | fórmula cerrada o gradiente | iterativo (lbfgs) | no busca: guarda y vota |
| Perilla del optimizador | — | η | `max_iter` | — |
| Perilla del modelo | `max_depth` | `alpha` | `C` (regularización) | `k` |

**Para la Sesión 6:** árboles y bosques sobre este mismo dataset, F1 y curva ROC para comparar modelos sin fijar un umbral, y la columna `duration` que dejamos por fuera hoy. Se entrega **E2**: línea base + un modelo supervisado sobre su dataset, con la métrica elegida y justificada.